In [2]:
!pip install openai langchain langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 48.7 MB/s eta 0:00:00


In [3]:
from getpass import getpass
import os

# 🔑 Set Groq API Key & Endpoint
os.environ["OPENAI_API_KEY"] = getpass("Enter your Token: ")
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

Enter your Token: ··········


In [10]:
import json
import os
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory, VectorStoreRetrieverMemory
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import AIMessage, HumanMessage

# --- 1. GROQ SETUP --- os.environ["OPENAI_API_KEY"] = "your_groq_api_key" #
# 🔐 Replace with your Groq API key
# os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model="llama3-70b-8192",  # ✅ or 'llama3-70b-8192'
    temperature=0.3
)

# --- 2. MEMORY SETUP ---
embedding = HuggingFaceEmbeddings()

chat_memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True
)

vector_store = FAISS.from_texts(["example event"], embedding)
event_memory = VectorStoreRetrieverMemory(
    retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
    memory_key="events"
)

# --- 3. SUMMARIZER ---
def summarize_events(events):
    if not events:
        return "No events detected."
    flat = "\n".join(f"{e['timestamp']}: {e['event_type']} at {e['location']} (confidence: {e['confidence']})" for e in events)
    prompt = f"Summarize the following surveillance events clearly and concisely:\n\n{flat}"
    return llm.invoke(prompt)

# --- 4. CHAT HANDLER ---
def handle_chat(user_input, summary_text):
    event_memory.save_context({"input": "event_summary"}, {"output": summary_text})
    chat_memory.chat_memory.add_user_message(user_input)

    retrieved = event_memory.load_memory_variables({"input": user_input})
    chat_history = chat_memory.load_memory_variables({"input": user_input}).get("chat_history", [])

    formatted = "\n".join([
        f"User: {msg.content}" if isinstance(msg, HumanMessage) else f"Assistant: {msg.content}"
        for msg in chat_history
    ])

    prompt = f"""
You are an intelligent assistant helping summarize and analyze security surveillance data.

Context:
{retrieved.get("events", "")}

Conversation History:
{formatted}

User: {user_input}
Assistant:"""

    response = llm.invoke(prompt)
    chat_memory.chat_memory.add_ai_message(response)
    return response



/tmp/ipython-input-1416018144.py:19: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()


In [12]:
if __name__ == "__main__":
    events = [
        {"timestamp": "2025-08-04T10:00:00", "event_type": "unauthorized_entry", "confidence": 0.91, "location": "Zone A"},
        {"timestamp": "2025-08-04T10:03:00", "event_type": "loitering", "confidence": 0.79, "location": "Zone B"},
        {"timestamp": "2025-08-04T10:05:00", "event_type": "tailgating", "confidence": 0.88, "location": "Zone A"}
    ]

    print("🔎 Generating Summary...")
    summary = summarize_events(events)
    print("📄 Summary:\n", summary)

    print("\n💬 Ask a question about the events:")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            break
        reply = handle_chat(user_input, summary)
        print("Assistant:", reply.content)


🔎 Generating Summary...
📄 Summary:
 content='Here is a clear and concise summary of the surveillance events:\n\nOn August 4, 2025, the following events were detected:\n\n* 10:00 AM: Unauthorized entry into Zone A (91% confidence)\n* 10:03 AM: Loitering in Zone B (79% confidence)\n* 10:05 AM: Tailgating in Zone A (88% confidence)' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 103, 'total_tokens': 184, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_time': 0.002907012, 'completion_time': 0.24375279, 'total_time': 0.246659802}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_bf16903a67', 'finish_reason': 'stop', 'logprobs': None} id='run--8e9d1a41-be4a-43a6-b806-e34579f76592-0'

💬 Ask a question about the events:
You: can you example me in more easy and shprt way
Assistant: Here is a shorter and easier-to-read summary of the surveillance events:

**August 4, 2025**

* 10:00 AM: Unauthorized entry into